# Integration of financial and stagnation data

This notebook aims to integrate dataframe with financial metric and stagnation metric together. Then generate a stagnation score and final score using the formula:

- 0.7 * stagnation_score + 0.3 * lbo_financial_score

Then show the final result. The actual score may varies from the presentation as there is minor changes in logic of score computation


In [40]:
import pandas as pd
import numpy as np
import openpyxl


fin_metric_df = pd.read_excel("lbo_finscreening_reordered.xlsx")
fin_metric_df.head(5)

,lbo_rank,ticker,year,sector,missing_critical_data,Unnamed: 5,pass_fcf_filter,pass_interest_coverage_filter,pass_debt_to_ebitda_filter,pass_current_ratio_filter,...,Unnamed: 36,capex_to_revenue,capex_to_ebitda,capex_to_revenue_score,capex_to_ebitda_score,capex_burden_score,Unnamed: 42,lbo_financial_score,lbo_status,lbo_rating
0,1,APP,2025,Communication Services,False,NaN,True,True,True,True,...,NaN,0.005167,0.006503,5.0,5.0,5.0,NaN,98.0,Pass,Ideal
1,2,ACN,2025,Technology,False,NaN,True,True,True,True,...,NaN,0.008612,0.050562,5.0,4.0,4.5,NaN,96.5,Pass,Ideal
2,3,AMD,2025,Technology,False,NaN,True,True,True,True,...,NaN,0.028119,0.133883,4.0,3.0,3.5,NaN,93.5,Pass,Ideal
3,4,AME,2025,Industrials,False,NaN,True,True,True,True,...,NaN,0.017598,0.056571,5.0,4.0,4.5,NaN,93.0,Pass,Ideal
4,5,ADP,2025,Technology,False,NaN,True,True,True,True,...,NaN,0.026604,0.086163,4.0,4.0,4.0,NaN,91.5,Pass,Ideal


In [41]:
stag_metric_df = pd.read_csv("sp500_stagnation_metric_df.csv", index_col=0)

# Remove duplicate tickers 
stag_metric_df = stag_metric_df.drop_duplicates(subset=['ticker'], keep='first')

def is_empty_for_removal(val):
    """Return True if val is considered empty for removal rules."""
    # NaN or None
    if pd.isna(val):
        return True
    # Empty dict or string "{}"
    if isinstance(val, dict) and not val:
        return True
    if isinstance(val, str) and val.strip() == "{}":
        return True
    return False

cols_to_check = [
    'innovation_freq',
    'innovation_decay_slope',
    'topic_rigidity',
    'sentiment_growth_correlation'
]

mask_all_empty = (
    stag_metric_df['innovation_freq'].apply(is_empty_for_removal) &
    stag_metric_df['innovation_decay_slope'].apply(is_empty_for_removal) &
    stag_metric_df['topic_rigidity'].apply(is_empty_for_removal) &
    stag_metric_df['sentiment_growth_correlation'].apply(is_empty_for_removal)
)

stag_metric_df = stag_metric_df[~mask_all_empty]

# reset index for cleanliness
stag_metric_df = stag_metric_df.reset_index(drop=True)

stag_metric_df.head(5)

,ticker,innovation_freq,innovation_decay_slope,topic_rigidity,topic_similarities,strategic_freq,strategic_decay_slope,latest_year,leverage_tolerance,debt_capacity,cash_flow_health,liquidity,growth_indicators,overall_feasibility,sentiment_by_year,capex_growth,sentiment_growth_correlation
0,MO,"{2025: 0.004451733833177133, 2026: 0.004419682...",NaN,0.997791,[np.float32(0.9977911)],"{2025: 0.004113297927730917, 2026: 0.003776819...",NaN,2025,{'interest_coverage': np.float64(8.97706032285...,"{'additional_debt_capacity_usd': None, 'assump...",{'free_cash_flow_usd': np.float64(9074000000.0...,{'current_ratio': np.float64(0.648022722307188...,{'revenue_growth_yoy': np.float64(-0.014918802...,Feasible – strong financial health,NaN,NaN,NaN
1,APP,"{2023: 0.007352389526596143, 2024: 0.006349482...",-0.000657,0.882868,"[np.float32(0.78179944), np.float32(0.9094051)...","{2023: 0.008552779653387351, 2024: 0.007915108...",-0.000093,2025,{'interest_coverage': np.float64(20.0947076554...,"{'additional_debt_capacity_usd': None, 'assump...",{'free_cash_flow_usd': np.float64(3942776000.0...,{'current_ratio': np.float64(3.321961211226971...,{'revenue_growth_yoy': np.float64(0.6999436734...,Feasible – strong financial health,"{2023: np.float64(0.5018092580504862), 2024: n...","{2024: np.float64(0.6001032880013771), 2025: n...",-1.000000
2,ACN,"{2022: 0.00409626216077829, 2023: 0.0037412314...",0.000022,0.992622,"[np.float32(0.9819153), np.float32(0.99837655)...","{2022: 0.007851169141491723, 2023: 0.008885424...",-0.000258,2025,{'interest_coverage': np.float64(47.4327317275...,"{'additional_debt_capacity_usd': None, 'assump...",{'free_cash_flow_usd': np.float64(10874360000....,{'current_ratio': np.float64(1.420034947750101...,{'revenue_growth_yoy': np.float64(0.0736020532...,Feasible – strong financial health,"{2022: np.float64(0.5716042012252189), 2023: n...","{2023: np.float64(0.26438235203997784), 2024: ...",-0.959882
3,AMD,"{2024: 0.006064162754303599, 2025: 0.007740585...",-0.000093,0.996713,"[np.float32(0.9954549), np.float32(0.99797124)]","{2024: 0.003129890453834116, 2025: 0.004184100...",-0.000371,2025,{'interest_coverage': np.float64(32.6030534351...,"{'additional_debt_capacity_usd': None, 'assump...",{'free_cash_flow_usd': np.float64(6735000000.0...,{'current_ratio': np.float64(2.850026441036489...,{'revenue_growth_yoy': np.float64(0.3433779329...,Feasible – strong financial health,"{2024: np.float64(0.5949717783058683), 2025: n...",{2025: np.float64(-0.5314465408805031)},NaN
4,ADBE,"{2023: 0.005222981116914423, 2024: 0.005125201...",0.000177,0.984074,"[np.float32(0.98305565), np.float32(0.97440505...","{2023: 0.0064282844515869825, 2024: 0.00732171...",-0.000067,2025,{'interest_coverage': np.float64(34.2091254752...,"{'additional_debt_capacity_usd': None, 'assump...",{'free_cash_flow_usd': np.float64(9852000000.0...,{'current_ratio': np.float64(0.996372549019607...,{'revenue_growth_yoy': np.float64(0.1052778423...,Feasible – strong financial health,"{2023: np.float64(0.5828006468140162), 2024: n...","{2024: np.float64(0.49166666666666664), 2025: ...",-1.000000


In [42]:
import pandas as pd
import numpy as np
import ast
import re

def extract_latest_from_dict_str(series):
    """
    Convert a column of string-dicts (e.g. "{2025: 0.00445}") into a numeric series
    containing the value for the largest year key.
    Returns NaN for invalid or missing entries.
    """
    def get_latest(val):
        if pd.isna(val):
            return np.nan
        if isinstance(val, dict):
            d = val
        elif isinstance(val, str):
            # Try to safely evaluate the string as a dict
            try:
                # Replace np.float64() etc. with plain numbers
                cleaned = re.sub(r'np\.float64\(([^)]+)\)', r'\1', val)
                d = ast.literal_eval(cleaned)
                if not isinstance(d, dict):
                    return np.nan
            except:
                return np.nan
        else:
            return np.nan
        
        # Find the maximum key (year) that is numeric
        years = [int(k) for k in d.keys() if str(k).lstrip('-').isdigit()]
        if not years:
            return np.nan
        latest_year = max(years)
        return float(d[latest_year])
    
    return series.apply(get_latest)

def compute_stagnation_score(df):
    """
    Compute a stagnation score (0 = dynamic, 1 = stagnant) for each firm.
    Expected columns:
        - ticker
        - innovation_freq (string dict or dict) -> latest year value
        - innovation_decay_slope (numeric)
        - topic_rigidity (numeric)
        - sentiment_growth_correlation (numeric)
    """
    # Work on a copy
    df_score = df.copy()
    
    # 1. Extract latest innovation frequency
    df_score['innov_freq_scalar'] = extract_latest_from_dict_str(df_score['innovation_freq'])
    
    # 2. Ensure numeric conversion for other columns
    numeric_cols = {
        'innov_decay_slope': 'innovation_decay_slope',
        'topic_rigidity': 'topic_rigidity',
        'sent_corr': 'sentiment_growth_correlation'
    }
    for new_col, orig_col in numeric_cols.items():
        df_score[new_col] = pd.to_numeric(df_score[orig_col], errors='coerce')
    
    # 3. Select rows with all four metrics present
    required = ['innov_freq_scalar', 'innov_decay_slope', 'topic_rigidity', 'sent_corr']
    df_clean = df_score.dropna(subset=required).copy()
    
    if df_clean.empty:
        df_score['stagnation_score'] = np.nan
        return df_score
    
    # 4. Min-max normalization helper
    def normalize(series):
        min_val = series.min()
        max_val = series.max()
        if min_val == max_val:
            return pd.Series([0.5] * len(series), index=series.index)
        return (series - min_val) / (max_val - min_val)
    
    # 5. Compute stagnation contribution for each metric
    # innovation_freq: lower -> more stagnation
    freq_norm = normalize(df_clean['innov_freq_scalar'])
    df_clean['stagn_freq'] = 1 - freq_norm
    
    # innovation_decay_slope: more negative -> more stagnation
    slope_norm = normalize(df_clean['innov_decay_slope'])
    df_clean['stagn_slope'] = 1 - slope_norm
    
    # topic_rigidity: higher -> more stagnation
    rig_norm = normalize(df_clean['topic_rigidity'])
    df_clean['stagn_rigidity'] = rig_norm
    
    # sentiment_growth_correlation: lower -> more stagnation
    corr_norm = normalize(df_clean['sent_corr'])
    df_clean['stagn_corr'] = 1 - corr_norm
    
    # 6. Average the four components
    stagnation_cols = ['stagn_freq', 'stagn_slope', 'stagn_rigidity', 'stagn_corr']
    df_clean['stagnation_score'] = df_clean[stagnation_cols].mean(axis=1).clip(0, 1)
    
    # 7. Merge back to original dataframe
    df_score['stagnation_score'] = df_clean['stagnation_score']
    
    return df_score


df_with_score = compute_stagnation_score(stag_metric_df)


df_filtered = df_with_score.dropna(subset=['stagnation_score'])

# Check result
print(f"Rows before: {len(df_with_score)}")
print(f"Rows after: {len(df_filtered)}")
print(df_filtered[['ticker', 'stagnation_score']].sort_values('stagnation_score', ascending=False).head(10))

Rows before: 334
Rows after: 235
    ticker  stagnation_score
179   LDOS          0.897121
166   JKHY          0.864845
51     CCI          0.860915
121      F          0.846095
203   MRNA          0.840684
118   FITB          0.839672
78     ESS          0.837595
79     FDS          0.831286
312    UHS          0.825545
276    SPG          0.823753


In [43]:
# Merge on ticker (inner join keeps only tickers present in both)
merged = pd.merge(df_filtered[['ticker', 'stagnation_score']], 
                  fin_metric_df[['ticker', 'lbo_financial_score']], 
                  on='ticker', how='inner')

# Rescale lbo_financial_score to 0-1
merged['lbo_score_normalized'] = merged['lbo_financial_score'] / 100.0

# Compute final score
merged['final_score'] = 0.7 * merged['stagnation_score'] + 0.3 * merged['lbo_score_normalized']

# Sort by final_score (lower = better)
merged_sorted = merged.sort_values('stagnation_score', ascending=True)
merged_sorted.head(30)

,ticker,stagnation_score,lbo_financial_score,lbo_score_normalized,final_score
16,AMGN,0.428746,68.0,0.680,0.504122
15,ADI,0.440342,90.5,0.905,0.579740
13,APD,0.464204,0.0,0.000,0.324943
5,ANET,0.475254,NaN,NaN,NaN
10,ALGN,0.502847,81.5,0.815,0.596493
14,ABT,0.518388,90.0,0.900,0.632872
12,ABBV,0.542002,85.0,0.850,0.634401
11,AWK,0.587590,4.5,0.045,0.424813
9,APH,0.590506,86.5,0.865,0.672854
3,AOS,0.609562,88.5,0.885,0.692193
